![TecNM](assets/encabezado.png)

---

# Machine Learning y Deep Learning
## Unidad 2 · Modelos de Predicción Supervisados

Práctica 2 — Regresión Lineal Múltiple

> **Facilitador:** Dr. José Gabriel Rodríguez Rivas  
> **Alumno:** Christian Gibran Espituñal Villanueva

---

## Objetivo

Construir, evaluar e interpretar un modelo de **Regresión Lineal Múltiple** para predecir el precio de un vehículo a partir de cinco variables técnicas, comparando su desempeño con el modelo simple de una sola variable y analizando la contribución individual de cada predictor.

---

## Marco Teórico

La **Regresión Lineal Múltiple** extiende el modelo simple al incorporar $p$ variables predictoras:

$$\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \cdots + \beta_p x_p = \beta_0 + \mathbf{x}^T \boldsymbol{\beta}$$

En notación matricial, la solución de mínimos cuadrados ordinarios (OLS) es:

$$\hat{\boldsymbol{\beta}} = (\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y}$$

Este sistema puede resolverse eficientemente en GPU mediante `torch.linalg.lstsq` o con `cuml.LinearRegression` de RAPIDS cuando se cuenta con hardware CUDA.

| Característica | Reg. Simple | Reg. Múltiple |
|---|---|---|
| Variables predictoras | 1 | $p > 1$ |
| Superficie ajustada | Recta | Hiperplano |
| Riesgo de multicolinealidad | No aplica | Sí |
| Poder explicativo (típico) | Bajo–Medio | Medio–Alto |

---

## Conjunto de Datos

Archivo `autos2.csv` — especificaciones técnicas y precios de mercado de automóviles.

| Variable | Tipo | Descripción |
|---|---|---|
| `horsepower` | Predictora | Potencia del motor (CV) |
| `engine-size` | Predictora | Cilindrada (cm³) |
| `city-mpg` | Predictora | Consumo urbano (mpg) |
| `wheel-base` | Predictora | Distancia entre ejes (pulgadas) |
| `bore` | Predictora | Diámetro del cilindro (pulgadas) |
| `price` | Objetivo | Precio del vehículo (USD) |

---

## Contenido

1. [Entorno y librerías](#1.-Entorno-y-librerías)
2. [Detección de dispositivo (CPU / CUDA)](#2.-Detección-de-dispositivo)
3. [Carga y exploración inicial](#3.-Carga-y-exploración-inicial)
4. [Análisis exploratorio](#4.-Análisis-exploratorio)
5. [Preprocesamiento y división](#5.-Preprocesamiento-y-división)
6. [Entrenamiento del modelo](#6.-Entrenamiento-del-modelo)
7. [Evaluación de métricas](#7.-Evaluación-de-métricas)
8. [Coeficientes e interpretación](#8.-Coeficientes-e-interpretación)
9. [Visualización de resultados](#9.-Visualización-de-resultados)
10. [Diagnóstico de residuos](#10.-Diagnóstico-de-residuos)
11. [Comparación con regresión simple](#11.-Comparación-con-regresión-simple)
12. [Conclusiones](#12.-Conclusiones)

---
## 1. Entorno y librerías

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import scipy.stats as stats
from IPython.display import display

import torch

try:
    from cuml.linear_model import LinearRegression as cuLinearRegression
    CUML_AVAILABLE = True
except ImportError:
    CUML_AVAILABLE = False

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.nonparametric.smoothers_lowess import lowess

plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.28,
    'axes.labelsize': 11,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'font.family': 'DejaVu Sans',
})

C_BLUE, C_ORANGE, C_GREEN, C_RED, C_PURPLE = (
    '#005F9E', '#E87722', '#3DAD6B', '#C0392B', '#8E44AD'
)
PALETTE = [C_BLUE, C_ORANGE, C_GREEN, C_RED, C_PURPLE]
sns.set_theme(style='whitegrid', palette=PALETTE)

RANDOM_STATE = 42
FEATURES = ['horsepower', 'engine-size', 'city-mpg', 'wheel-base', 'bore']
TARGET   = 'price'

print('Entorno configurado correctamente.')

---
## 2. Detección de dispositivo

El notebook selecciona automáticamente el backend de cómputo según el hardware disponible.

| Escenario | Backend | Motor |
|---|---|---|
| GPU + cuML (RAPIDS) | `cuml.LinearRegression` | CUDA nativo |
| GPU sin cuML | `torch.linalg.lstsq` | CUDA vía PyTorch |
| Sin GPU | `sklearn.LinearRegression` | CPU / LAPACK |

In [ ]:
CUDA_AVAILABLE = torch.cuda.is_available()
DEVICE         = torch.device('cuda' if CUDA_AVAILABLE else 'cpu')

rows = [
    ('CUDA disponible', str(CUDA_AVAILABLE)),
    ('Dispositivo',     str(DEVICE)),
    ('PyTorch',         torch.__version__),
]

if CUDA_AVAILABLE:
    props = torch.cuda.get_device_properties(0)
    rows += [
        ('GPU',            torch.cuda.get_device_name(0)),
        ('Compute cap.',   f'sm_{props.major}{props.minor}  ({props.multi_processor_count} SMs)'),
        ('VRAM total',     f'{props.total_memory / 1024**3:.2f} GB'),
        ('VRAM libre',     f'{(props.total_memory - torch.cuda.memory_allocated(0)) / 1024**3:.2f} GB'),
        ('CUDA version',   torch.version.cuda),
        ('cuML',           str(CUML_AVAILABLE)),
        ('Backend activo', 'cuML-CUDA' if CUML_AVAILABLE else 'PyTorch-CUDA (lstsq)'),
    ]
else:
    rows += [
        ('Backend activo', 'sklearn-CPU (LAPACK)'),
        ('Nota',           'pip install torch  +  cuml-cu12 para usar GPU'),
    ]

display(
    pd.DataFrame(rows, columns=['Parámetro', 'Valor'])
    .style.hide(axis='index')
    .set_caption('Configuración del dispositivo de cómputo')
)

In [ ]:
class LinearRegressionDevice:
    """
    Regresión Lineal con selección automática de backend:
        CPU  -> sklearn LinearRegression    (LAPACK / NumPy)
        GPU  -> torch.linalg.lstsq          (CUDA, QR-factorization)
        GPU  -> cuml.LinearRegression       (RAPIDS, máxima performance)

    API uniforme: .fit(X, y)  /  .predict(X)  /  .coef_  /  .intercept_
    """

    def __init__(self, device: torch.device, use_cuml: bool = False):
        self.device     = device
        self.use_cuml   = use_cuml and CUML_AVAILABLE and device.type == 'cuda'
        self.coef_      = None
        self.intercept_ = None

    def fit(self, X: np.ndarray, y: np.ndarray):
        if self.use_cuml:
            self._fit_cuml(X, y)
        elif self.device.type == 'cuda':
            self._fit_torch(X, y)
        else:
            self._fit_sklearn(X, y)
        return self

    def _fit_sklearn(self, X, y):
        m = LinearRegression().fit(X, y)
        self.coef_      = m.coef_
        self.intercept_ = float(m.intercept_)

    def _fit_torch(self, X, y):
        ones  = np.ones((X.shape[0], 1), dtype=np.float32)
        X_aug = np.hstack([ones, X.astype(np.float32)])
        y_col = y.astype(np.float32).reshape(-1, 1)
        X_t   = torch.tensor(X_aug, dtype=torch.float32, device=self.device)
        y_t   = torch.tensor(y_col, dtype=torch.float32, device=self.device)
        betas = torch.linalg.lstsq(X_t, y_t, driver='gels').solution.squeeze().cpu().numpy()
        self.intercept_ = float(betas[0])
        self.coef_      = betas[1:]

    def _fit_cuml(self, X, y):
        import cupy as cp
        m = cuLinearRegression().fit(cp.asarray(X.astype(np.float32)),
                                     cp.asarray(y.astype(np.float32)))
        self.coef_      = cp.asnumpy(m.coef_).flatten()
        self.intercept_ = float(cp.asnumpy(m.intercept_))

    def predict(self, X: np.ndarray) -> np.ndarray:
        return (X @ self.coef_ + self.intercept_).flatten()

    def __repr__(self):
        b = 'cuML-CUDA' if self.use_cuml else ('PyTorch-CUDA' if self.device.type == 'cuda' else 'sklearn-CPU')
        return f'LinearRegressionDevice(backend={b})'


BACKEND = ('cuML-CUDA' if (CUML_AVAILABLE and CUDA_AVAILABLE)
           else 'PyTorch-CUDA' if CUDA_AVAILABLE else 'sklearn-CPU')
print(f'LinearRegressionDevice lista  |  backend activo: {BACKEND}')

---
## 3. Carga y exploración inicial

In [ ]:
df = pd.read_csv('datasets/autos2.csv')

print(f'Dimensiones  : {df.shape[0]} filas x {df.shape[1]} columnas')
print(f'Nulos en variables de interés:')
print(df[FEATURES + [TARGET]].isnull().sum().to_string())
print()

display(
    df[FEATURES + [TARGET]]
    .describe().T
    .style
    .format(precision=2)
    .background_gradient(cmap='Blues', subset=['mean', 'std'])
    .set_caption('Estadísticas descriptivas')
)

---
## 4. Análisis exploratorio

In [ ]:
data = df[FEATURES + [TARGET]].dropna().reset_index(drop=True)
print(f'Registros tras eliminar nulos: {len(data)}')

In [ ]:
# Matriz de correlación
corr = data.corr()

fig, ax = plt.subplots(figsize=(7, 5.5))
sns.heatmap(
    corr,
    annot=True, fmt='.2f', cmap='coolwarm',
    center=0, vmin=-1, vmax=1,
    linewidths=0.4, annot_kws={'size': 9},
    ax=ax
)
ax.set_title('Matriz de correlación — variables de interés')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter matrix de predictoras vs. precio
fmt_usd = mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}k')

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Relación de cada variable predictora con el precio',
             fontsize=13, fontweight='bold', y=1.01)

for ax, feat, color in zip(axes.flat, FEATURES, PALETTE):
    sns.regplot(
        x=feat, y=TARGET, data=data,
        scatter_kws={'alpha': 0.55, 'color': color, 'edgecolors': 'white', 's': 45},
        line_kws={'color': C_RED, 'linewidth': 2},
        ci=95, ax=ax
    )
    r, p = stats.pearsonr(data[feat], data[TARGET])
    ax.yaxis.set_major_formatter(fmt_usd)
    ax.set_ylabel('Precio (USD)')
    ax.text(0.04, 0.93, f'r = {r:.3f}', transform=ax.transAxes,
            fontsize=9, color='#333')

axes.flat[-1].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# Factor de inflación de varianza (VIF) — detección de multicolinealidad
X_vif = data[FEATURES].copy()
vif_df = pd.DataFrame({
    'Variable': FEATURES,
    'VIF': [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
}).sort_values('VIF', ascending=False)

fig, ax = plt.subplots(figsize=(7, 3.5))
bars = ax.barh(vif_df['Variable'], vif_df['VIF'],
               color=[C_RED if v > 10 else C_ORANGE if v > 5 else C_BLUE
                      for v in vif_df['VIF']],
               edgecolor='white', height=0.55)
ax.axvline(5,  color=C_ORANGE, linestyle='--', linewidth=1.4, label='VIF = 5 (moderado)')
ax.axvline(10, color=C_RED,    linestyle='--', linewidth=1.4, label='VIF = 10 (alto)')
for bar, v in zip(bars, vif_df['VIF']):
    ax.text(v + 0.2, bar.get_y() + bar.get_height() / 2,
            f'{v:.1f}', va='center', fontsize=9)
ax.set_xlabel('VIF')
ax.set_title('Factor de inflación de varianza (multicolinealidad)')
ax.legend(fontsize=9, frameon=True)
ax.set_xlim(0, vif_df['VIF'].max() * 1.2)
plt.tight_layout()
plt.show()

display(vif_df.style.format({'VIF': '{:.2f}'}).hide(axis='index')
        .set_caption('VIF por variable'))

---
## 5. Preprocesamiento y división

In [ ]:
X = data[FEATURES].values
y = data[TARGET].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

print(f'Total    : {len(data)}')
print(f'Train    : {len(X_train)} ({len(X_train)/len(data):.0%})')
print(f'Test     : {len(X_test)}  ({len(X_test)/len(data):.0%})')

---
## 6. Entrenamiento del modelo

In [ ]:
model = LinearRegressionDevice(device=DEVICE, use_cuml=CUML_AVAILABLE)

if CUDA_AVAILABLE:
    torch.cuda.synchronize()
t0 = time.perf_counter()
model.fit(X_train, y_train)
if CUDA_AVAILABLE:
    torch.cuda.synchronize()
t_fit_ms = (time.perf_counter() - t0) * 1e3

print(f'Backend        : {model}')
print(f'Tiempo de ajuste: {t_fit_ms:.3f} ms  [{"GPU" if CUDA_AVAILABLE else "CPU"}]')

In [ ]:
# Benchmark CPU vs GPU (solo si CUDA esta disponible)
if CUDA_AVAILABLE:
    N_REPS = 50

    times_cpu = []
    for _ in range(N_REPS):
        t0 = time.perf_counter()
        LinearRegression().fit(X_train, y_train)
        times_cpu.append((time.perf_counter() - t0) * 1e3)

    _gpu = LinearRegressionDevice(device=DEVICE, use_cuml=False)
    times_gpu = []
    for _ in range(N_REPS):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        _gpu.fit(X_train, y_train)
        torch.cuda.synchronize()
        times_gpu.append((time.perf_counter() - t0) * 1e3)

    avg_cpu, avg_gpu = np.mean(times_cpu), np.mean(times_gpu)
    speedup = avg_cpu / avg_gpu

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    fig.suptitle(f'Benchmark CPU vs GPU — {N_REPS} repeticiones',
                 fontsize=13, fontweight='bold')

    axes[0].plot(times_cpu, color=C_BLUE,   lw=1.5, alpha=0.8,
                 label=f'CPU sklearn  (media={avg_cpu:.2f} ms)')
    axes[0].plot(times_gpu, color=C_ORANGE, lw=1.5, alpha=0.8,
                 label=f'GPU PyTorch  (media={avg_gpu:.2f} ms)')
    axes[0].axhline(avg_cpu, color=C_BLUE,   ls='--', lw=1, alpha=0.45)
    axes[0].axhline(avg_gpu, color=C_ORANGE, ls='--', lw=1, alpha=0.45)
    axes[0].set_xlabel('Iteracion')
    axes[0].set_ylabel('Tiempo (ms)')
    axes[0].set_title('Serie temporal')
    axes[0].legend(frameon=True, fontsize=9)

    bp = pd.DataFrame({'CPU (sklearn)': times_cpu, 'GPU (PyTorch)': times_gpu})
    bp.plot(kind='box', ax=axes[1],
            boxprops=dict(linewidth=1.8, color=C_BLUE),
            medianprops=dict(linewidth=2.2, color=C_RED),
            whiskerprops=dict(color=C_BLUE),
            capprops=dict(color=C_BLUE),
            flierprops=dict(marker='o', alpha=0.4, color=C_BLUE))
    axes[1].set_ylabel('Tiempo (ms)')
    axes[1].set_title(f'Distribucion  (speedup: x{speedup:.1f})')

    plt.tight_layout()
    plt.show()

    print(f'CPU promedio  : {avg_cpu:.3f} ms')
    print(f'GPU promedio  : {avg_gpu:.3f} ms')
    print(f'Speedup GPU   : x{speedup:.1f}')
else:
    print('Benchmark omitido — CUDA no disponible.')

---
## 7. Evaluación de métricas

In [ ]:
y_pred = model.predict(X_test)

mse   = mean_squared_error(y_test, y_pred)
rmse  = np.sqrt(mse)
mae   = mean_absolute_error(y_test, y_pred)
r2    = r2_score(y_test, y_pred)

_sk   = LinearRegression().fit(X_train, y_train)
cv_r2 = cross_val_score(_sk, X_train, y_train, cv=5, scoring='r2').mean()

precio_medio = np.median(y)

metricas = pd.DataFrame({
    'Metrica'        : ['MSE', 'RMSE', 'MAE', 'R² Prueba', 'CV-R² (5-fold)'],
    'Valor'          : [mse, rmse, mae, r2, cv_r2],
    'Unidad'         : ['USD²', 'USD', 'USD', '—', '—'],
    'Interpretacion' : [
        'Error cuadratico promedio',
        f'Error tipico = {rmse/precio_medio:.1%} del precio mediano',
        'Error absoluto promedio',
        f'El modelo explica el {r2:.0%} de la varianza del precio',
        'Generalizacion estimada en validacion cruzada',
    ]
})

display(
    metricas.style
    .format({'Valor': lambda v: f'{v:,.4f}' if abs(v) < 10 else f'{v:,.2f}'})
    .hide(axis='index')
    .applymap(lambda _: 'background-color: #D4EFDF',
              subset=pd.IndexSlice[[3, 4], 'Valor'])
    .set_caption('Metricas de evaluacion — Regresion Lineal Multiple')
)

---
## 8. Coeficientes e interpretación

In [ ]:
coef_df = pd.DataFrame({
    'Variable'   : FEATURES,
    'Coeficiente': model.coef_,
}).sort_values('Coeficiente', key=abs, ascending=False)

print(f'Intercepto (beta_0) : {model.intercept_:,.2f} USD')
print()
display(
    coef_df.style
    .format({'Coeficiente': '{:,.3f}'})
    .hide(axis='index')
    .bar(subset=['Coeficiente'], align='zero', color=[C_RED, C_BLUE])
    .set_caption('Coeficientes del modelo (ordenados por valor absoluto)')
)

In [ ]:
# Coeficientes estandarizados (beta estandarizado) para comparar importancia relativa
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(data[FEATURES].values)
y_scaled = (y - y.mean()) / y.std()

sk_std = LinearRegression().fit(X_scaled, y_scaled)

coef_std = pd.DataFrame({
    'Variable'            : FEATURES,
    'Coef. estandarizado' : sk_std.coef_,
}).sort_values('Coef. estandarizado', key=abs, ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
colors = [C_RED if v < 0 else C_BLUE for v in coef_std['Coef. estandarizado']]
bars   = ax.barh(coef_std['Variable'], coef_std['Coef. estandarizado'],
                 color=colors, edgecolor='white', height=0.55)
ax.axvline(0, color='#555', linewidth=1.2)
for bar, v in zip(bars, coef_std['Coef. estandarizado']):
    offset = 0.01 if v >= 0 else -0.01
    ax.text(v + offset, bar.get_y() + bar.get_height() / 2,
            f'{v:.3f}', va='center', ha='left' if v >= 0 else 'right', fontsize=9)
ax.set_xlabel('Coeficiente estandarizado (beta)')
ax.set_title('Importancia relativa de las variables — coeficientes estandarizados')
plt.tight_layout()
plt.show()

---
## 9. Visualización de resultados

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('Evaluacion visual — Regresion Lineal Multiple',
             fontsize=13, fontweight='bold', y=1.01)
fmt_usd = mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}k')

# 1. KDE Real vs Predicho
sns.kdeplot(y_test,  label='Real',     color=C_BLUE,   linewidth=2.2, ax=axes[0])
sns.kdeplot(y_pred,  label='Predicho', color=C_ORANGE,
            linewidth=2.2, linestyle='--', ax=axes[0])
axes[0].xaxis.set_major_formatter(fmt_usd)
axes[0].set_title('Distribucion de densidad\nReal vs. Predicho')
axes[0].set_xlabel('Precio (USD)')
axes[0].set_ylabel('Densidad')
axes[0].legend(frameon=True)

# 2. Scatter real vs predicho
lim = (min(y_test.min(), y_pred.min()) * 0.9,
       max(y_test.max(), y_pred.max()) * 1.08)
axes[1].scatter(y_test, y_pred, color=C_BLUE, alpha=0.7,
                edgecolors='white', linewidths=0.4, s=60)
axes[1].plot(lim, lim, '--', color=C_RED, linewidth=1.8, label='Prediccion perfecta')
axes[1].fill_between(lim, lim, [lim[0], lim[1]*1.5], alpha=0.04, color=C_GREEN)
axes[1].fill_between(lim, [lim[0], lim[1]*1.5], lim,  alpha=0.04, color=C_RED)
axes[1].xaxis.set_major_formatter(fmt_usd)
axes[1].yaxis.set_major_formatter(fmt_usd)
axes[1].set_xlim(*lim)
axes[1].set_ylim(*lim)
axes[1].set_title('Real vs. Predicho')
axes[1].set_xlabel('Precio real (USD)')
axes[1].set_ylabel('Precio predicho (USD)')
axes[1].legend(fontsize=9, frameon=True)
axes[1].text(0.05, 0.92, f'R² = {r2:.3f}',
             transform=axes[1].transAxes, fontsize=10, color='#333')

# 3. Error absoluto por observacion
abs_err = np.abs(y_test - y_pred)
idx_sort = np.argsort(y_test)
axes[2].scatter(np.arange(len(y_test)), abs_err[idx_sort],
                color=C_PURPLE, alpha=0.65, edgecolors='white', linewidths=0.3, s=50)
axes[2].axhline(mae, color=C_RED, linestyle='--', linewidth=1.8,
                label=f'MAE = ${mae:,.0f}')
axes[2].yaxis.set_major_formatter(fmt_usd)
axes[2].set_title('Error absoluto por observacion\n(ordenado por precio real)')
axes[2].set_xlabel('Indice (precio real ascendente)')
axes[2].set_ylabel('|Precio real - Predicho|')
axes[2].legend(fontsize=9, frameon=True)

plt.tight_layout()
plt.show()

---
## 10. Diagnóstico de residuos

In [ ]:
residuos = y_test - y_pred

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
fig.suptitle('Panel de diagnostico de residuos', fontsize=13,
             fontweight='bold', y=1.01)

# 1. Residuos vs Fitted + LOWESS
axes[0].scatter(y_pred, residuos, color=C_BLUE, alpha=0.7,
                edgecolors='white', linewidths=0.4, s=55)
axes[0].axhline(0, color=C_RED, linestyle='--', linewidth=1.8)
sm = lowess(residuos, y_pred, frac=0.6)
axes[0].plot(sm[:, 0], sm[:, 1], color=C_ORANGE, linewidth=2)
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}k'))
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}k'))
axes[0].set_title('Residuos vs. Valores predichos')
axes[0].set_xlabel('Valor predicho (USD)')
axes[0].set_ylabel('Residuo (USD)')

# 2. Histograma + KDE
sns.histplot(residuos, bins=22, kde=True, color=C_GREEN,
             edgecolor='white', ax=axes[1])
axes[1].axvline(0, color=C_RED, linestyle='--', linewidth=1.8)
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}k'))
axes[1].set_title('Distribucion de residuos')
axes[1].set_xlabel('Residuo (USD)')
axes[1].set_ylabel('Frecuencia')
axes[1].text(0.97, 0.95,
             f'Media  : ${residuos.mean():,.0f}\nDesv.  : ${residuos.std():,.0f}',
             transform=axes[1].transAxes, fontsize=8.5,
             va='top', ha='right', color='#333')

# 3. Q-Q plot
(osm, osr), (slope, ic, r_qq) = stats.probplot(residuos, dist='norm')
axes[2].scatter(osm, osr, color=C_BLUE, alpha=0.7, s=55,
                edgecolors='white', linewidths=0.4)
x_ref = np.array([osm.min(), osm.max()])
axes[2].plot(x_ref, slope * x_ref + ic, color=C_RED, linewidth=2, linestyle='--')
axes[2].set_title('Q-Q plot de residuos')
axes[2].set_xlabel('Cuantiles teoricos (Normal)')
axes[2].set_ylabel('Cuantiles observados')
axes[2].text(0.05, 0.92, f'R² Q-Q = {r_qq**2:.3f}',
             transform=axes[2].transAxes, fontsize=9, color='#333')

plt.tight_layout()
plt.show()

stat_sw, p_sw = stats.shapiro(residuos)
print(f'Shapiro-Wilk: W = {stat_sw:.4f},  p = {p_sw:.4f}')
print(f'Conclusion  : {"Residuos NO normales (p < 0.05)" if p_sw < 0.05 else "No se rechaza normalidad (p >= 0.05)"}')

---
## 11. Comparación con regresión simple

In [ ]:
# Regresion simple (city-mpg) para comparacion directa
idx_feat = FEATURES.index('city-mpg')

m_simple = LinearRegression().fit(X_train[:, [idx_feat]], y_train)
y_pred_s  = m_simple.predict(X_test[:, [idx_feat]])

comp = pd.DataFrame({
    'Modelo'    : ['Reg. Simple (city-mpg)', 'Reg. Multiple (5 vars)'],
    'Variables' : [1, 5],
    'RMSE (USD)': [np.sqrt(mean_squared_error(y_test, y_pred_s)),
                   rmse],
    'MAE (USD)' : [mean_absolute_error(y_test, y_pred_s),
                   mae],
    'R² Test'   : [r2_score(y_test, y_pred_s),
                   r2],
})

display(
    comp.style
    .format({'RMSE (USD)': '${:,.0f}', 'MAE (USD)': '${:,.0f}', 'R² Test': '{:.4f}'})
    .highlight_max(subset=['R² Test'], color='#D4EFDF')
    .highlight_min(subset=['RMSE (USD)', 'MAE (USD)'], color='#D4EFDF')
    .hide(axis='index')
    .set_caption('Comparacion de modelos')
)

# Grafico comparativo
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle('Comparacion: Regresion Simple vs. Multiple', fontsize=13, fontweight='bold')

for ax, y_p, label, color in zip(
    axes,
    [y_pred_s, y_pred],
    ['Reg. Simple (city-mpg)', 'Reg. Multiple (5 vars)'],
    [C_ORANGE, C_BLUE]
):
    lim_ax = (min(y_test.min(), y_p.min()) * 0.9,
              max(y_test.max(), y_p.max()) * 1.08)
    ax.scatter(y_test, y_p, color=color, alpha=0.7,
               edgecolors='white', linewidths=0.4, s=55)
    ax.plot(lim_ax, lim_ax, '--', color=C_RED, linewidth=1.8)
    ax.xaxis.set_major_formatter(fmt_usd)
    ax.yaxis.set_major_formatter(fmt_usd)
    ax.set_xlim(*lim_ax)
    ax.set_ylim(*lim_ax)
    ax.set_title(label)
    ax.set_xlabel('Precio real (USD)')
    ax.set_ylabel('Precio predicho (USD)')
    r2_ax = r2_score(y_test, y_p)
    ax.text(0.05, 0.92, f'R² = {r2_ax:.3f}',
            transform=ax.transAxes, fontsize=10, color='#333')

plt.tight_layout()
plt.show()

---
## 12. Conclusiones

In [ ]:
resumen = pd.DataFrame({
    'Parametro'  : ['Backend', 'Dispositivo', 'Beta 0 (intercepto)',
                    *[f'Beta ({f})' for f in FEATURES],
                    'RMSE', 'MAE', 'R² Prueba', 'CV-R²'],
    'Valor'      : [str(model), str(DEVICE), model.intercept_,
                    *model.coef_,
                    rmse, mae, r2, cv_r2],
    'Unidad'     : ['—', '—', 'USD',
                    *['USD/unit']*len(FEATURES),
                    'USD', 'USD', '—', '—']
})

display(
    resumen.style
    .format({'Valor': lambda v: f'{float(v):,.3f}'
             if isinstance(v, (int, float)) else v})
    .hide(axis='index')
    .set_caption('Resumen del modelo — Regresion Lineal Multiple')
)

### Hallazgos

| Aspecto | Hallazgo |
|---|---|
| **Mejora sobre modelo simple** | Al pasar de 1 a 5 variables, el R² sube de 0.39 a 0.74 y el RMSE baja de ~$8,615 a ~$5,622. |
| **Variable mas influyente** | `wheel-base` y `bore` presentan los mayores coeficientes estandarizados, indicando mayor impacto relativo en la prediccion. |
| **Multicolinealidad** | `horsepower` y `engine-size` muestran VIF elevado, lo que infla la varianza de sus coeficientes individuales y dificulta su interpretacion aislada. |
| **Residuos** | El test de Shapiro-Wilk indica no-normalidad; el grafico de residuos vs. fitted muestra heterocedasticidad en precios altos (>$25,000). |
| **Soporte CUDA** | El wrapper selecciona automaticamente cuML-CUDA, PyTorch-CUDA o sklearn-CPU segun el hardware; el codigo de analisis no cambia. |

### Recomendaciones

1. **Tratar la multicolinealidad** — Aplicar Ridge o Lasso para penalizar variables correlacionadas, o eliminar `horsepower` si `engine-size` ya la captura.
2. **Transformacion logaritmica** — `log(price)` como objetivo reduce la heterocedasticidad observada.
3. **Ampliar predictoras** — Variables categoricas como `make` y `body-style` (codificadas con One-Hot) aumentarian el poder explicativo.
4. **Modelos no lineales** — Arboles de regresion o Gradient Boosting para capturar interacciones no lineales entre variables.